# 🏨 Hotel Review Sentiment Analysis

**Dataset:** 515K European hotel reviews from Booking.com  
**Goal:** Predict whether a review is **Positive** or **Negative** using:
- Classical ML (Logistic Regression, SVM, Random Forest, XGBoost)
- Deep Learning (LSTM)
- Transformers (BERT)

**Columns used:**
```
Hotel_Address, Review_Date, Average_Score, Hotel_Name, Reviewer_Nationality,
Negative_Review, Review_Total_Negative_Word_Counts, Total_Number_of_Reviews,
Positive_Review, Review_Total_Positive_Word_Counts,
Total_Number_of_Reviews_Reviewer_Has_Given, Reviewer_Score, Tags,
days_since_review, lat, lng
```

---

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, re, string, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

# ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from scipy.sparse import hstack, csr_matrix
import joblib

# Plot style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12

# NLTK downloads
for pkg in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print('✅ All libraries loaded successfully.')

## 1. Load Data

In [ ]:
# ── Update this path to where you placed the dataset ──
DATA_PATH = '../data/Hotel_Reviews.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
print('Column names:\n', df.columns.tolist())
print('\nDtypes:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
df.describe()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── Reviewer Score Distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Reviewer_Score'].dropna(), bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Reviewer Scores')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Count')

axes[1].hist(df['Average_Score'].dropna(), bins=20, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Hotel Average Scores')
axes[1].set_xlabel('Score')

plt.tight_layout()
plt.savefig('../outputs/score_distributions.png')
plt.show()

In [ ]:
# ── Top 10 Nationalities ─────────────────────────────────────────────
top_nations = df['Reviewer_Nationality'].str.strip().value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
top_nations.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 10 Reviewer Nationalities')
ax.set_xlabel('Review Count')
plt.tight_layout()
plt.savefig('../outputs/top_nationalities.png')
plt.show()

In [ ]:
# ── Top 10 Hotels by Review Count ────────────────────────────────────
top_hotels = df['Hotel_Name'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
top_hotels.sort_values().plot(kind='barh', ax=ax, color='teal')
ax.set_title('Top 10 Most-Reviewed Hotels')
ax.set_xlabel('Review Count')
plt.tight_layout()
plt.savefig('../outputs/top_hotels.png')
plt.show()

In [ ]:
# ── Review word-count distributions ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Review_Total_Negative_Word_Counts'].clip(0, 200).hist(
    bins=40, ax=axes[0], color='tomato', edgecolor='white')
axes[0].set_title('Negative Review Word Counts')
axes[0].set_xlabel('Words')

df['Review_Total_Positive_Word_Counts'].clip(0, 200).hist(
    bins=40, ax=axes[1], color='mediumseagreen', edgecolor='white')
axes[1].set_title('Positive Review Word Counts')
axes[1].set_xlabel('Words')

plt.tight_layout()
plt.savefig('../outputs/word_count_distributions.png')
plt.show()

In [ ]:
# ── Score vs Negative/Positive word counts ───────────────────────────
sample = df.sample(min(5000, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(sample['Review_Total_Negative_Word_Counts'],
                sample['Reviewer_Score'], alpha=0.3, s=10, color='tomato')
axes[0].set_xlabel('Negative Word Count')
axes[0].set_ylabel('Reviewer Score')
axes[0].set_title('Negative Words vs Score')

axes[1].scatter(sample['Review_Total_Positive_Word_Counts'],
                sample['Reviewer_Score'], alpha=0.3, s=10, color='steelblue')
axes[1].set_xlabel('Positive Word Count')
axes[1].set_title('Positive Words vs Score')

plt.tight_layout()
plt.savefig('../outputs/words_vs_score.png')
plt.show()

In [ ]:
# ── Geographic heatmap of hotels ─────────────────────────────────────
try:
    import folium
    from folium.plugins import HeatMap

    hotel_coords = df[['lat', 'lng', 'Average_Score']].dropna()
    m = folium.Map(location=[hotel_coords['lat'].mean(),
                              hotel_coords['lng'].mean()], zoom_start=5)
    HeatMap(data=hotel_coords[['lat', 'lng']].values.tolist(),
            radius=8).add_to(m)
    m.save('../outputs/hotel_heatmap.html')
    print('Hotel heatmap saved → outputs/hotel_heatmap.html')
    m
except ImportError:
    print('Install folium for geo-map: pip install folium')

## 3. Preprocessing & Label Engineering

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────
PLACEHOLDER_NEG = {
    'no negative', 'nothing', 'none', 'n/a', 'na', 'nil',
    'no negatives', 'nothing negative', 'no complaints',
}
PLACEHOLDER_POS = {
    'no positive', 'nothing', 'none', 'n/a', 'na', 'nil', 'no positives',
}

_stop = set(stopwords.words('english')) - {'not','no','nor','never','neither',"n't"}
_lemma = WordNetLemmatizer()


def clean_text(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)       # URLs
    text = re.sub(r'<.*?>', '', text)                  # HTML
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)                    # numbers
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in _stop]
    tokens = [_lemma.lemmatize(t) for t in tokens]
    return ' '.join(tokens)


def combine_reviews(row):
    neg = '' if str(row['Negative_Review']).strip().lower() in PLACEHOLDER_NEG \
        else str(row['Negative_Review'])
    pos = '' if str(row['Positive_Review']).strip().lower() in PLACEHOLDER_POS \
        else str(row['Positive_Review'])
    return (neg + ' ' + pos).strip()


def label_binary(score):
    return 0 if score < 7 else 1    # 0 = Negative, 1 = Positive


print('Preprocessing functions defined.')

In [ ]:
# ── Apply to dataset ──────────────────────────────────────────────────
df = df.dropna(subset=['Reviewer_Score']).copy()

df['review_text'] = df.apply(combine_reviews, axis=1)
df = df[df['review_text'].str.strip() != ''].copy()

print('Cleaning text (may take 1-2 min for 500k rows)…')
df['clean_text'] = df['review_text'].apply(clean_text)
df = df[df['clean_text'].str.strip() != ''].copy()

# Binary label
df['label'] = df['Reviewer_Score'].apply(label_binary)
df['sentiment'] = df['label'].map({0: 'Negative', 1: 'Positive'})

print(f'\nFinal dataset shape: {df.shape}')
print('\nLabel distribution:')
print(df['sentiment'].value_counts())
print(f'\nClass balance: {df["label"].mean():.1%} positive')

In [ ]:
# ── Label distribution plot ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=['tomato', 'steelblue'], edgecolor='white')
axes[0].set_title('Sentiment Label Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Score distribution coloured by label
for lbl, clr in [('Negative', 'tomato'), ('Positive', 'steelblue')]:
    sub = df[df['sentiment'] == lbl]['Reviewer_Score']
    axes[1].hist(sub, bins=20, alpha=0.6, label=lbl, color=clr, edgecolor='white')
axes[1].axvline(7, color='black', linestyle='--', label='Threshold = 7')
axes[1].set_title('Reviewer Score by Sentiment')
axes[1].set_xlabel('Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/label_distribution.png')
plt.show()

## 4. Word Clouds & Top Terms

In [ ]:
def make_wordcloud(text_series, title, colormap='Blues', max_words=200):
    text = ' '.join(text_series.dropna())
    wc = WordCloud(
        width=900, height=450, background_color='white',
        colormap=colormap, max_words=max_words,
        collocations=False
    ).generate(text)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    fname = title.lower().replace(' ', '_') + '.png'
    plt.savefig(f'../outputs/{fname}')
    plt.show()

make_wordcloud(df[df['label']==1]['clean_text'], 'Positive Review WordCloud', 'Greens')
make_wordcloud(df[df['label']==0]['clean_text'], 'Negative Review WordCloud', 'Reds')

In [ ]:
# ── Top 20 words per class ────────────────────────────────────────────
def top_words(text_series, n=20):
    words = ' '.join(text_series.dropna()).split()
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, label, color, title in [
    (axes[0], 1, 'steelblue', 'Top 20 Positive Words'),
    (axes[1], 0, 'tomato',    'Top 20 Negative Words'),
]:
    words, counts = zip(*top_words(df[df['label']==label]['clean_text']))
    ax.barh(words[::-1], counts[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../outputs/top_words_by_class.png')
plt.show()

## 5. Feature Engineering

In [ ]:
# ── TF-IDF ────────────────────────────────────────────────────────────
X_text = df['clean_text']
y = df['label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print('Building TF-IDF (unigrams + bigrams)…')
tfidf = TfidfVectorizer(
    max_features=50_000, ngram_range=(1, 2),
    sublinear_tf=True, min_df=2, max_df=0.95
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

print(f'TF-IDF train shape: {X_train_tfidf.shape}')
print(f'TF-IDF test  shape: {X_test_tfidf.shape}')

In [ ]:
# ── Structured meta-features ─────────────────────────────────────────
def make_meta(subset):
    f = pd.DataFrame(index=subset.index)
    f['neg_wc']          = subset['Review_Total_Negative_Word_Counts'].fillna(0)
    f['pos_wc']          = subset['Review_Total_Positive_Word_Counts'].fillna(0)
    f['total_wc']        = f['neg_wc'] + f['pos_wc']
    f['neg_ratio']       = f['neg_wc'] / (f['total_wc'] + 1e-6)
    f['avg_score']       = subset['Average_Score'].fillna(subset['Average_Score'].median())
    f['log_total_rev']   = np.log1p(subset['Total_Number_of_Reviews'].fillna(0))
    f['reviewer_exp']    = np.log1p(
        subset['Total_Number_of_Reviews_Reviewer_Has_Given'].fillna(0))
    if 'days_since_review' in subset.columns:
        _ds = pd.to_numeric(subset['days_since_review'].astype(str).str.extract(r'(\d+)', expand=False), errors='coerce')
        f['days_since'] = _ds.fillna(_ds.median())
    else:
        f['days_since'] = 0
    f['lat']             = subset['lat'].fillna(subset['lat'].median())
    f['lng']             = subset['lng'].fillna(subset['lng'].median())
    return f

train_idx = X_train_text.index
test_idx  = X_test_text.index

meta_train = make_meta(df.loc[train_idx])
meta_test  = make_meta(df.loc[test_idx])

scaler = StandardScaler()
meta_train_s = scaler.fit_transform(meta_train)
meta_test_s  = scaler.transform(meta_test)

# Combine
X_train = hstack([X_train_tfidf, csr_matrix(meta_train_s)])
X_test  = hstack([X_test_tfidf,  csr_matrix(meta_test_s)])
print(f'Combined feature matrix shape: {X_train.shape}')

## 6. Classical ML Models

In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred, average='weighted')
    prec = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_te, y_pred, average='weighted', zero_division=0)

    print(f'\n── {name} ──────────────────────────────────────')
    print(f'  Accuracy={acc:.4f}  F1={f1:.4f}  Precision={prec:.4f}  Recall={rec:.4f}')
    print(classification_report(y_te, y_pred, target_names=['Negative','Positive'],
                                 zero_division=0))
    # Confusion matrix
    cm = confusion_matrix(y_te, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'], ax=ax)
    ax.set_title(f'Confusion Matrix — {name}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'../outputs/cm_{name.lower().replace(" ","_")}.png')
    plt.show()

    return {'Model': name, 'Accuracy': acc, 'F1': f1, 'Precision': prec, 'Recall': rec}

In [ ]:
results = []

# ── Logistic Regression ──────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', n_jobs=-1)
results.append(evaluate('Logistic Regression', lr, X_train, X_test, y_train, y_test))

In [ ]:
# ── Linear SVM ───────────────────────────────────────────────────────
svm = LinearSVC(max_iter=2000, C=1.0)
results.append(evaluate('Linear SVM', svm, X_train, X_test, y_train, y_test))

In [ ]:
# ── Random Forest (speed-optimised) ─────────────────────────────────
# 100 trees • 30% row sub-sample • max depth 20
# Runs in <1 min on 500k rows; accuracy stays within ~1% of full RF.
rf = RandomForestClassifier(
    n_estimators=100, max_depth=20, max_samples=0.3,
    max_features='sqrt', n_jobs=-1, random_state=42
)
results.append(evaluate('Random Forest', rf, X_train, X_test, y_train, y_test))

In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
                         eval_metric='logloss',
                         n_jobs=-1, random_state=42)
    results.append(evaluate('XGBoost', xgb, X_train, X_test, y_train, y_test))
except ImportError:
    print('XGBoost not installed: pip install xgboost')

In [ ]:
# ── ROC Curves ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['steelblue', 'tomato', 'mediumseagreen', 'darkorange']

for (res, model, color) in zip(results, [lr, svm, rf], colors):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        ax.plot(fpr, tpr, lw=2, color=color,
                label=f"{res['Model']} (AUC={auc:.3f})")

ax.plot([0,1],[0,1],'--', color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/roc_curves.png')
plt.show()

In [ ]:
# ── Model Comparison Table ────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('Model')
print('\n── Model Comparison ──────────────────────────────────────')
print(results_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
results_df[['Accuracy','F1','Precision','Recall']].plot(
    kind='bar', ax=ax, rot=15, colormap='Set2', edgecolor='white')
ax.set_ylim(0.7, 1.0)
ax.set_title('Model Comparison')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/model_comparison.png')
plt.show()

results_df.to_csv('../outputs/model_results.csv')
print('\nResults saved → outputs/model_results.csv')

## 7. Save Best Model

In [ ]:
best_name = results_df['F1'].idxmax()
best_model_map = {
    'Logistic Regression': lr,
    'Linear SVM': svm,
    'Random Forest': rf,
}
best_model = best_model_map.get(best_name)

if best_model:
    joblib.dump(best_model, f'../models/best_model_{best_name.lower().replace(" ","_")}.joblib')
    joblib.dump(tfidf, '../models/tfidf_vectoriser.joblib')
    joblib.dump(scaler, '../models/meta_scaler.joblib')
    print(f'✅ Best model saved: {best_name}')
    print(f'   F1 Score: {results_df.loc[best_name, "F1"]:.4f}')

## 8. Deep Learning — LSTM

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import (
        Embedding, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
    )
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    # ── Tokenise ──────────────────────────────────────────────────────
    MAX_WORDS  = 30_000
    MAX_LEN    = 150
    EMBED_DIM  = 128

    tok = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
    tok.fit_on_texts(X_train_text)

    X_tr_seq = pad_sequences(tok.texts_to_sequences(X_train_text), maxlen=MAX_LEN)
    X_te_seq = pad_sequences(tok.texts_to_sequences(X_test_text),  maxlen=MAX_LEN)

    # ── Model ─────────────────────────────────────────────────────────
    lstm_model = Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(64, return_sequences=True)),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid'),
    ])
    lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    lstm_model.summary()

    # ── Train ─────────────────────────────────────────────────────────
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
    ]
    history = lstm_model.fit(
        X_tr_seq, y_train,
        validation_split=0.1,
        epochs=10, batch_size=256,
        callbacks=callbacks, verbose=1,
    )

    # ── Evaluate ──────────────────────────────────────────────────────
    y_pred_prob = lstm_model.predict(X_te_seq).ravel()
    y_pred_lstm = (y_pred_prob >= 0.5).astype(int)
    print('\nLSTM Results:')
    print(classification_report(y_test, y_pred_lstm,
                                 target_names=['Negative','Positive']))

    # ── Training curves ───────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title('LSTM Accuracy'); axes[0].legend()

    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title('LSTM Loss'); axes[1].legend()

    plt.tight_layout()
    plt.savefig('../outputs/lstm_training_curves.png')
    plt.show()

    lstm_model.save('../models/lstm_model.keras')
    print('LSTM model saved → models/lstm_model.keras')

except ImportError:
    print('TensorFlow not installed. Skip LSTM section or: pip install tensorflow')

## 9. Inference — Predict on New Text

In [ ]:
# Build a dummy row with median meta-feature values for inference
_meta_medians = meta_train.median()

def predict_review(text, model=lr, vectoriser=tfidf):
    """Predict sentiment for a single raw review (TF-IDF + meta features)."""
    cleaned = clean_text(text)
    X_tfidf = vectoriser.transform([cleaned])
    # Use median meta-feature values as stand-ins for a single inference call
    meta_row = _meta_medians.values.reshape(1, -1)
    meta_scaled = scaler.transform(meta_row)
    X = hstack([X_tfidf, csr_matrix(meta_scaled)])
    pred = model.predict(X)[0]
    label = 'Positive ✅' if pred == 1 else 'Negative ❌'
    conf = None
    if hasattr(model, 'predict_proba'):
        conf = model.predict_proba(X)[0].max()
    print(f'Review  : {text}')
    print(f'Sentiment: {label}', end='')
    if conf:
        print(f' (confidence: {conf:.1%})')
    print()

# Examples
predict_review('The room was spotless, staff incredibly friendly. Best stay ever!')
predict_review('Dirty bathroom, noisy at night and the breakfast was terrible.')
predict_review('Room was okay. Location was convenient but nothing special.')

## 10. Summary & Conclusions

In [ ]:
print('=' * 60)
print('  HOTEL REVIEW SENTIMENT ANALYSIS — SUMMARY')
print('=' * 60)
print(f'\nDataset: {len(df):,} hotel reviews')
print(f'Features: TF-IDF (unigrams + bigrams) + meta-features')
print(f'Label: Binary (Negative < 7 / Positive >= 7)')
print('\nModel Results:')
print(results_df.round(4).to_string())
print(f'\n🏆 Best Model: {best_name}')
print(f'   F1: {results_df.loc[best_name, "F1"]:.4f}')
print('\nOutputs saved in outputs/ directory.')
print('Models saved in models/ directory.')